# SLP (NMSE)

This notebook computes the normalized mean square error of atmospheric surface pressure.
It is compared to ERA5 observations, as well as the CESM2 large ensemble and CMIP6 model output.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from cupid_utils.atm import nmse
from cupid_utils.atm import seasonal_climatology_weighted

## Parameters

These variables are set in `config.yml`

In [ ]:
CESM_output_dir = []
case_names = []
start_dates = []
end_dates = []
ts_dir = None
obs_data_dir = ""
validation_path = ""
regridded_output = []

In [ ]:
# # Want some base case parameter defaults to equal control case values
if ts_dir is None:
    ts_dir = CESM_output_dir

## Read in the current case

In [ ]:
def fix_time_dim(dat):
    """CESM2 output sets time as the end of the averaging interval (e.g. January average is midnight on February 1st);
    This function sets the time dimension to the midpoint of the averaging interval.
    Note that CESM3 output sets time to the midpoint already, so this function should not change CESM3 data.
    """
    if "time" not in dat.dims:
        return dat
    if "bounds" not in dat.time.attrs:
        return dat
    time_bounds_avg = dat[dat.time.attrs["bounds"]].mean("nbnd")
    time_bounds_avg.attrs = dat.time.attrs
    dat = dat.assign_coords({"time": time_bounds_avg})
    return xr.decode_cf(dat)

In [ ]:
cmap = plt.get_cmap("Set1")
colors = cmap.colors[: len(case_names)]

case_info = {}
for case_name, start_date, end_date, ts_dir, regridded_output in zip(
    case_names, start_dates, end_dates, ts_dir, regridded_output
):
    if regridded_output:
        file_path = f"{ts_dir}/{case_name}/atm/proc/tseries/regrid"
    else:
        file_path = f"{ts_dir}/{case_name}/atm/proc/tseries"
    print(file_path)
    case_info[case_name] = {
        "file_path": file_path,
        "start_date": start_date,
        "end_date": end_date,
        "color": colors[case_names.index(case_name)],
    }

In [ ]:
for case_name, case_data in case_info.items():
    dat = (
        fix_time_dim(
            xr.open_mfdataset(f"{case_data["file_path"]}/*PSL*.nc", decode_times=False)
        )
        .sel(time=slice(case_data["start_date"], case_data["end_date"]))
        .PSL
        / 100.0
    )

    case_info[case_name]["dat"] = dat

# Ensure all datasets have the same coordinates as the output data
# (Avoid round-off level differences since all data should be on the same grid)
base_case = case_names[0]
base_data = case_info[base_case]

lon = base_data["dat"].lon.data
lat = base_data["dat"].lat.data

In [ ]:
# --Compute seasonal and annual means
for case_name, case_data in case_info.items():
    case_info[case_name]["dat"] = seasonal_climatology_weighted(case_data["dat"]).load()

## Read in validation data and other CMIP models for comparison (precomputed)

In [ ]:
# ---ERA5
era5 = xr.open_dataset(
    os.path.join(obs_data_dir, validation_path, "PSL_ERA5.nc")
).assign_coords({"lon": lon, "lat": lat})
era5 = era5 / 100.0  # convert to hPa

# ---CESM2
lens2 = xr.open_dataset(
    os.path.join(obs_data_dir, validation_path, "PSL_LENS2.nc")
).assign_coords({"lon": lon, "lat": lat})
lens2 = lens2 / 100.0  # convert to hPa

# ---CMIP6
modelfiles = sorted(
    glob.glob(f"{os.path.join(obs_data_dir,validation_path)}/CMIP6/*.nc")
)
datcmip6 = [
    xr.open_dataset(ifile).assign_coords({"lon": lon, "lat": lat}).mean("M")
    for ifile in modelfiles
]
datcmip6 = xr.concat(datcmip6, dim="model")
datcmip6 = datcmip6 / 100.0

## Compute the NMSE

In [ ]:
nmse_cesm2 = []
nmse_cmip6 = []
for ivar in era5.data_vars:
    nmse_cesm2.append(nmse(era5[ivar], lens2[ivar]))
    nmse_cmip6.append(nmse(era5[ivar], datcmip6[ivar]))

nmse_cesm2 = xr.merge(nmse_cesm2)
nmse_cmip6 = xr.merge(nmse_cmip6)

for case_name, case_data in case_info.items():
    nmse_dat = []
    dat = case_data["dat"]
    for ivar in era5.data_vars:
        nmse_dat.append(nmse(era5[ivar], dat[ivar]))

    nmse_dat = xr.merge(nmse_dat)
    case_info[case_name]["nmse_dat"] = nmse_dat

### Set up the plot panel

In [ ]:
def plotnmse(fig, cmip6, cesm2, seasonal_key, x1, x2, y1, y2, titlestr):
    ax = fig.add_axes([x1, y1, x2 - x1, y2 - y1])

    cmip6 = cmip6.sortby(cmip6, ascending=False)
    binedges = np.arange(0, cmip6.size, 1)
    ax.bar(
        binedges,
        cmip6,
        width=1,
        bottom=0,
        edgecolor="black",
        color="gray",
        label="CMIP6",
    )

    for case_name, case_data in case_info.items():
        nmse_dat = case_data["nmse_dat"][seasonal_key]
        ax.plot(
            cmip6.size + 1,
            nmse_dat,
            "x",
            color=case_data["color"],
            label=f"{case_name}",
        )

    ax.fill_between(
        np.arange(0, cmip6.size + 3, 1) - 0.5,
        np.arange(0, cmip6.size + 3, 1) * 0 + np.array(cesm2.min()),
        np.arange(0, cmip6.size + 3, 1) * 0 + np.array(cesm2.max()),
        color="salmon",
        alpha=0.5,
        label="LENS2",
    )

    ax.set_xlim(-0.5, cmip6.size + 2 - 0.5)
    ax.set_xticks([])
    ax.set_ylabel("NMSE", fontsize=14)
    ax.set_title(titlestr, fontsize=16)

    ax.legend()

    return ax

In [ ]:
fig = plt.figure(figsize=(16, 16))

vert_coord = 0.99
for case_name, case_data in case_info.items():
    fig.text(
        0.5,
        vert_coord,
        "CASE = "
        + case_name
        + " "
        + case_data["start_date"]
        + " to "
        + case_data["end_date"],
        ha="center",
        va="center",
        fontsize=14,
        color=case_data["color"],
    )
    vert_coord = vert_coord - 0.015

fig.text(
    0.5,
    vert_coord,
    "Other runs = 1979-01-01 to 2023-12-31",
    ha="center",
    va="center",
    fontsize=14,
)
vert_coord = vert_coord - 0.015

fig.text(
    0.5,
    vert_coord,
    "Validation data = ERA5 1979-01-01 to 2023-12-31",
    ha="center",
    va="center",
    fontsize=14,
)

vert_coord = vert_coord - 0.03

ax = plotnmse(
    fig,
    nmse_cmip6["AM"],
    nmse_cesm2["AM"],
    "AM",
    0.3,
    0.7,
    vert_coord - 0.16,
    vert_coord,
    "NMSE, SLP, AM",
)
ax = plotnmse(
    fig,
    nmse_cmip6["DJF"],
    nmse_cesm2["DJF"],
    "DJF",
    0.05,
    0.45,
    vert_coord - 0.36,
    vert_coord - 0.21,
    "NMSE, SLP, DJF",
)
ax = plotnmse(
    fig,
    nmse_cmip6["MAM"],
    nmse_cesm2["MAM"],
    "MAM",
    0.55,
    0.95,
    vert_coord - 0.36,
    vert_coord - 0.21,
    "NMSE, SLP, MAM",
)
ax = plotnmse(
    fig,
    nmse_cmip6["JJA"],
    nmse_cesm2["JJA"],
    "JJA",
    0.05,
    0.45,
    vert_coord - 0.56,
    vert_coord - 0.41,
    "NMSE, SLP, JJA",
)
ax = plotnmse(
    fig,
    nmse_cmip6["SON"],
    nmse_cesm2["SON"],
    "SON",
    0.55,
    0.95,
    vert_coord - 0.56,
    vert_coord - 0.41,
    "NMSE, SLP, SON",
)